# Introduction 

-  **This notebook for making Data analysis and Machine Learning model for the "Item_Outlet_Sales".**
- **first, Data Analysis processing was made to clean, extract, and visualise data in order to understand it properly.**
- **Then, i haaved used 3 models to predcit the sales [Linear Regression, DecisionTreeRegressor, RandomForestRegressor]**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

### sklearn 
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelBinarizer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline


## linear regression model
from sklearn.linear_model import LinearRegression, Lasso
#DecisionTreeRegressor model
from sklearn.tree import DecisionTreeRegressor
#RandomForestRegressor
from sklearn.ensemble import RandomForestRegressor
# Fine-Tune using GridSearchCV
from sklearn.model_selection import GridSearchCV

#cross-validation
from sklearn.model_selection import cross_val_score

import os

In [ ]:
#importing data 
"D:\D\machinlearning\projects\Bigmart Sales"

train = pd.read_csv("/kaggle/input/bigmart-sales-data/Train.csv")
test = pd.read_csv("/kaggle/input/bigmart-sales-data/Test.csv")

In [ ]:
# shape of the data
train.shape, test.shape

#  Exploratory Data Analaysis

In [ ]:
train.info()

In [ ]:
test.info()

In [ ]:
train.describe()

In [ ]:
for i in train.select_dtypes(exclude='object'):
    sns.boxplot(y=train[i])
    plt.show()

In [ ]:
sns.histplot(train['Item_Outlet_Sales'], kde=True, color="red",)

In [ ]:
train.describe(include='O')

#### Insights

- there are outliers in ["item_visibility", "Sales"]
- missing values in 2 columns only ['Item_Weight' ,'Outlet_Size']
- there are right skewness in the sales ditribution 
- "Item_Visibility" has minimum value 0, which cannot be true because the visibility has to be more than 0

#  Data Cleansing


In [ ]:
#chcking duplicates
print(train.duplicated().value_counts())
print(test.duplicated().value_counts())

In [ ]:
#check the null values in "outlet_size"
train[train["Outlet_Size"].isnull() ==True]

#### checking values related to "outlet", trying to extract the missed values 

In [ ]:
train[train["Outlet_Size"].isnull() ==True]['Outlet_Location_Type'].value_counts()

In [ ]:
train[train["Outlet_Size"].isnull() ==True]['Outlet_Identifier'].value_counts()

In [ ]:
train[train["Outlet_Size"].isnull() ==True]['Outlet_Type'].value_counts()

#### - from the above analysis, it's clearly that ['OUT045', 'OUT017'] are both >> Tier 2 and Supermarket type 1
### and 
#### -  ['OUT010'] is >> Tier3 and Grocery Store 

### so we are going to search about the value of "Outlet_Size" that oftten shown with these 2 values in both of the listed above cases

In [ ]:
## for both ['OUT045', 'OUT017']
train[(train["Outlet_Location_Type"] =="Tier 2" ) & (train["Outlet_Type"] =="Supermarket Type1")]['Outlet_Size'].value_counts()

In [ ]:
train[(train['Outlet_Size'].isnull() ==True)]

In [ ]:
## so for ['OUT010']
train[(train["Outlet_Location_Type"] =="Tier 3" ) & (train["Outlet_Type"] =="Grocery Store")]['Outlet_Size'].value_counts()

## there are no values have the both loctaion_type and type of the outlet so we should find any other column refer to it



In [ ]:
train[train["Outlet_Type"] == "Grocery Store"]['Outlet_Size'].value_counts()

In [ ]:
train[(train['Outlet_Location_Type'] == "Tier 3") ]['Outlet_Size'].value_counts()

#### based on the 2 previuos commands, it appears that every "Grocery store" in the dataframe is "Small" size 
#### however that, the size of "Tier 3" always medium or high
> so i will assume that it's medium since it's the mode value before the last editing that we made on the previous nulls

In [ ]:
#so "Outlet_Size" of ['OUT045', 'OUT017'] will be >> "Small"
train.loc[(train['Outlet_Size'].isnull() ==True) &(train['Outlet_Identifier'] != 'OUT010'),"Outlet_Size"] = "Small"


## for other outlets identifier they will be "Medium"
train.loc[(train['Outlet_Size'].isnull() ==True) &(train['Outlet_Identifier'] == 'OUT010'),"Outlet_Size"] = "Medium"

- dealing with the null values in 'item weight'

In [ ]:
sns.histplot(train['Item_Weight'],kde=True)

In [ ]:
train[train['Item_Weight'].isnull()]

In [ ]:
### using sklearn preprocessing to impute null values with mean
imputer = SimpleImputer()
Item_Weight = imputer.fit_transform(train['Item_Weight'].values.reshape(-1,1))


In [ ]:
sns.histplot(Item_Weight,kde=True)

In [ ]:
### tryin another approach for handling missing in item weights
cols = ['Item_Visibility','Item_MRP','Item_Weight']
x = train[cols]

impute_it = IterativeImputer()
x = impute_it.fit_transform(x)


In [ ]:
sns.histplot(x[:,2],kde=True)

In [ ]:
## so we will go wtih the second approach
train['Item_Weight'] = x[:,2]
train.info()

In [ ]:
## change the minimum values of "Item_Visibility" by giving them mean value
train.loc[train['Item_Visibility']==0, 'Item_Visibility'] = train['Item_Visibility'].mean()

In [ ]:
train

In [ ]:
#extract number of years for each outlet 
train['Outlet_Age'] = 2019 - train['Outlet_Establishment_Year']
train['Outlet_Age']

In [ ]:
#showing the mean sales for each outlet with the other categories columns related to outlet
outlet_cols = ['Outlet_Size', 'Outlet_Location_Type','Outlet_Type']
for i in outlet_cols:
    plt.figure(figsize=(12,10))
    sns.barplot(x='Outlet_Identifier', y= 'Item_Outlet_Sales', hue=i,data=train)
    plt.show()

In [ ]:
## check the age for every outlet
#df_out = train[["Outlet_Identifier",'Outlet_Age','Item_Outlet_Sales']]
#df_out.set_index('Outlet_Identifier')

df_out = train.groupby(['Outlet_Identifier','Outlet_Age'],as_index=False)['Item_Outlet_Sales'].mean().sort_values(by='Item_Outlet_Sales')
df_out.set_index('Outlet_Identifier',inplace=True)
#scaling mean sales by dividing on 100
df_out['Item_Outlet_Sales'] = df_out['Item_Outlet_Sales']/100

df_out

In [ ]:
plt.figure(figsize=(12,9))

plt.plot(df_out.index, df_out.Outlet_Age, "-b", label="Average Outlet Sales")
plt.bar(df_out.index, df_out.Item_Outlet_Sales , width=0.5,alpha=0.8, color='green', label="Average percipitation mm", )

#### then age has not a significant factor on sales

# Correlation

In [ ]:
### check colleration for all columns
train_corr = train.corr()
mask = np.triu(np.ones_like(train_corr,dtype=bool))

plt.figure(figsize=(13,10))
sns.heatmap(train_corr, cmap='RdYlGn_r', mask=mask , annot=True)
plt.xticks(rotation=65)
plt.show()


- there are no significant colleration with the sales except for the "Item_MRP" 


In [ ]:
plt.scatter(train['Item_Outlet_Sales'], train['Item_MRP'])

In [ ]:
### changing outliers to lower and maximum values
upper = train['Item_Outlet_Sales'].quantile(0.95)
lower = train['Item_Outlet_Sales'].quantile(0.05)

train['Item_Outlet_Sales'] = train['Item_Outlet_Sales'].apply(lambda x: upper if x >upper 
                                                              else(lower if x < lower else x))

In [ ]:
## visualising the sales after getting rid of outliers
sns.boxplot(y='Item_Outlet_Sales',data=train)

In [ ]:
plt.scatter(train['Item_Outlet_Sales'], train['Item_MRP'])

#### Data Exploratory for the items columns


In [ ]:
train

In [ ]:
for i in train.columns:
    print(i)
    print(train[i].value_counts())
    print("*********************\n")


### Note!!
- "Item_Fat_Content" >> has same values with different labels need to be adjusted
- [HINT FROM PREVIOUS WORK IN KAGGLE (hiralmshah)]"Item_Identifier" of each item start with ['fd' or 'dr' or 'nc'] >> can be used as item type identifier for FOOD , Drink, Non-Consumable 
- drop useless columns


In [ ]:
## adjusting values of fat_content
train['Item_Fat_Content']= train['Item_Fat_Content'].apply(lambda x: "Regular" if x=='reg' or x == 'Regular' else 'Low Fat')


In [ ]:
## item identifier classification
train['Item_Identifier'].str[:2].value_counts()

In [ ]:
## exctracting items_type_classification column from Item_identitfier
train['Item_ID_Type']=train['Item_Identifier'].apply(lambda x: "Food" if x[:2] =='FD' 
                                               else( "Drink" if x[:2]=='DR' else "Non-Consumable") )
train['Item_ID_Type'].value_counts()

In [ ]:
## delete "Outlet_Establishment_Year" column
train.drop(columns=['Outlet_Establishment_Year','Item_Type'], inplace=True)

In [ ]:
train.info()

### start changing categories data to numerical

In [ ]:
### using labelEncoder for ['Item_Identifier','Outlet_Identifier']
lencod = LabelEncoder()
ids = ['Item_Identifier','Outlet_Identifier','Item_Fat_Content','Outlet_Size','Outlet_Location_Type','Outlet_Type','Item_ID_Type']
for i in ids:
    train[i]=lencod.fit_transform(train[i])
train

In [ ]:
### using get_dummies to make one hot encoding for ther other categorical columns
lb = OneHotEncoder()
cat_cols = ['Item_Fat_Content','Outlet_Size','Outlet_Location_Type','Outlet_Type','Item_ID_Type']

train = pd.get_dummies(train,columns=cat_cols)
    
train.info()

# Pipline
- for the test dataframe to apply all changes that we done to train df

### create pipline function for test df

In [ ]:
def pipline(in_df):
    #so "Outlet_Size" of ['OUT045', 'OUT017'] will be >> "Small" and last one will be  'Medium'
    in_df.loc[(in_df['Outlet_Size'].isnull() ==True) &(in_df['Outlet_Identifier'] != 'OUT010'),"Outlet_Size"] = "Small"
    in_df.loc[(in_df['Outlet_Size'].isnull() ==True) &(in_df['Outlet_Identifier'] == 'OUT010'),"Outlet_Size"] = "Medium"
    
    ### tryin another approach for handling missing in item weights
    cols = ['Item_Visibility','Item_MRP','Item_Weight']
    x = in_df[cols]
    impute_it = IterativeImputer()
    x = impute_it.fit_transform(x)
    in_df['Item_Weight'] = x[:,2]
    
    ## change the minimum values of "Item_Visibility" by giving them mean value
    in_df.loc[in_df['Item_Visibility']==0, 'Item_Visibility'] = in_df['Item_Visibility'].mean()
    
    #extract number of years for each outlet 
    in_df['Outlet_Age'] = 2019 - in_df['Outlet_Establishment_Year']
    

    ## adjusting values of fat_content
    in_df['Item_Fat_Content']= in_df['Item_Fat_Content'].apply(lambda x: "Regular" if x=='reg' or x == 'Regular' else 'Low Fat')
    
    ## exctracting items_type_classification column from Item_identitfier
    in_df['Item_ID_Type']=in_df['Item_Identifier'].apply(lambda x: "Food" if x[:2] =='FD' 
                                               else( "Drink" if x[:2]=='DR' else "Non-Consumable") )
    
    ## delete "Outlet_Establishment_Year" and 'Item_Type' columns
    in_df.drop(columns=['Outlet_Establishment_Year','Item_Type'], inplace=True)
    
    
    
    #start encoding columns
    ##start with labelencoding
    ids = ['Item_Identifier','Outlet_Identifier','Item_Fat_Content','Outlet_Size'
           ,'Outlet_Location_Type','Outlet_Type','Item_ID_Type']
    for i in ids:
        in_df[i]=lencod.fit_transform(in_df[i])
        
    
    ### using get_dummies to make one hot encoding for ther other categorical columns
    cat_cols = ['Item_Fat_Content','Outlet_Size','Outlet_Location_Type','Outlet_Type','Item_ID_Type']
    in_df = pd.get_dummies(in_df,columns=cat_cols)
    
    out_df = in_df.copy()
    return out_df

In [ ]:
mod_test = pipline(test)
mod_test.info()

# start creating ML model

In [ ]:
# split target vector
y = train['Item_Outlet_Sales'].copy()
X = train.drop(columns='Item_Outlet_Sales')

### first will split train to 4 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=20)

X_train.shape , y_train.shape

# linear regression model 

In [ ]:

lg = LinearRegression()

lg.fit(X_train, y_train)

In [ ]:
#checking the score
print(f"Linear Regression Score: {lg.score(X_train, y_train)}")

**- it's low score but can be a start for tuning the model**

*** now will get the y_pred and compare error with y_test**

In [ ]:
y_pred = lg.predict(X_test)
y_pred

In [ ]:
y_test

In [ ]:
# checking the mean squared error and root mean squared error
mse = mean_squared_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)

print(f"Mean Squared Error for Regression: {mse}")
print(f"Root Mean Squared Error: {rmse}")

#### well, is not a satisfying error but the range of median values of set item range between 850 to 3100.
- so this model need to be adjusted by standerizing the numerical columns using Standerization 

- Now i will proceed with different model (DecisionTreeRegressor)

# DecisionTreeRegressor

In [ ]:
tree_reg =DecisionTreeRegressor()

tree_reg.fit(X_train, y_train)

#checking score 
tree_reg.score(X_train, y_train)

In [ ]:
y_pred_tree = tree_reg.predict(X_test)
tree_mse = mean_squared_error(y_test, y_pred_tree)
tree_rmse = np.sqrt(tree_mse)
tree_rmse

#### cannot use "DecisionTreeRegressor" as it overfitting

#### using RandomForestRegressor model 

# RandomForestRegressor model

In [ ]:
forest_reg = RandomForestRegressor()

forest_reg.fit(X_train, y_train)

#checking score
print(f"score of Random Forest Regressor model: {forest_reg.score(X_train, y_train)}")


In [ ]:
y_pred_forest = forest_reg.predict(X_test)
forest_mse = mean_squared_error(y_test, y_pred_forest)
forest_rmse = np.sqrt(forest_mse)

print(f"Root Mean Squared Error: {forest_rmse}")

### - the error of linear regression model still the lowest between the other models.

- however that, i will use Cross Validation with 10 folds to check these scores 

# Evaluation both models using Cross-Validation

In [ ]:
lrg_scores = cross_val_score(lg, X_train, y_train,scoring="neg_mean_squared_error", cv=10)
lrg_rmse_scores = np.sqrt(-lrg_scores)

In [ ]:
tree_scores = cross_val_score(tree_reg, X_train, y_train,scoring="neg_mean_squared_error", cv=10)
tree_rmse_scores = np.sqrt(-tree_scores)


In [ ]:
def display_scores(scores):
    print("Scores: ", scores)
    print("\nMean: ", scores.mean())
    print("Standard deviation: ", scores.std())


In [ ]:
#showing scores of linear regression model
display_scores(lrg_rmse_scores)

In [ ]:
#showing scores of  model DecisionTreeRegressor
display_scores(tree_rmse_scores)

In [ ]:
# get the scores validation using cross-validation
forest_scores = cross_val_score(forest_reg, X_train, y_train,scoring="neg_mean_squared_error", cv=10)
forest_rmse_scores = np.sqrt(-forest_scores)
# scores for RandomForestRegressor
display_scores(forest_rmse_scores)

# Fine-Tuning the models

#### Using GridSearchCV

- the following code searches for the best combi‐nation of hyperparameter values for the RandomForestRegressor

In [ ]:
param_grid = [
{'n_estimators': [3, 10, 30], 'max_features': [2, 4, 6, 8]},
{'bootstrap': [False], 'n_estimators': [3, 10], 'max_features': [2, 3, 4]},
]

grid_search = GridSearchCV(forest_reg, param_grid, cv=5,
scoring='neg_mean_squared_error')
grid_search.fit(X_train, y_train)

In [ ]:
#### now showing the best combinations parameters
grid_search.best_params_

- so the best parameters for forest regressor is [{'max_features': 6, 'n_estimators': 30}]
- the following code will show the all scores for each combined parameters and as we shown before we will use [{'max_features': 6, 'n_estimators': 30}]

In [ ]:
#showing the error score 
cvres = grid_search.cv_results_
for mean_score, params in zip(cvres["mean_test_score"], cvres["params"]):
    print(np.sqrt(-mean_score), params)

- the linear regression model still have the best error value

### from the previous 3 models, i will go with the LinearRegression model because it gave me the lowest error value with slight difference than the  RandomForestRegressor

# Predicting new values
#### now we will use it to predict the required column from test dataframe

In [ ]:
### first we will scaling the test data
scaler = StandardScaler()
scaler.fit(mod_test)
stnd_test = scaler.transform(mod_test)
stnd_test = pd.DataFrame(data =stnd_test, columns=mod_test.columns )


In [ ]:
##with mod_test data
predicted = lg.predict(stnd_test)

predicted

In [ ]:
stnd_test

# Visualising the Models

In [ ]:
model_scores = sorted([lrg_rmse_scores.mean(), tree_rmse_scores.mean(), forest_rmse_scores.mean()])
plt.figure(figsize=(13,10))


sns.set_style("darkgrid")
sns.barplot(x=['Linear Regression','Decision Tree Regressor','Random Forest Regressor'],y=model_scores,)

# Summary

- we managed to build a Linear Regression model with
    - Score: 0.58,
    - Root Mean Squared Error :  1002.14 